# 02 — Drug Safety Checker Pipeline

Standalone drug safety checking — can run at prescription time without
a full prediction pipeline. Uses PyHealth's ATC code system and DDI data.

In [ ]:
from pyhealth_enterprise.pipelines.drug_safety_checker import DrugSafetyChecker

# Without DDI adjacency — basic interaction check
checker = DrugSafetyChecker()

# Example patient medication list (ATC codes)
patient_meds = ['A02BC01', 'B01AC06', 'C07AB03', 'C10AA01', 'B01AA03']

interactions = checker.check_interactions(patient_meds)
print(f'Checking {len(patient_meds)} medications ({len(interactions)} pairs):')
print(interactions)

In [ ]:
# With DDI adjacency from a drug recommendation task dataset
# (run after drug_recommendation task setup)
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth.tasks import drug_recommendation_mimic3_fn

ds = SyntheticEHRDataset(); ds.load()
task_dataset = ds.dataset.set_task(drug_recommendation_mimic3_fn)

checker_with_ddi = DrugSafetyChecker(ddi_adj=task_dataset.ddi_adj)
high_risk = checker_with_ddi.flag_high_risk_combinations(patient_meds, ddi_threshold=0.3)
print(f'High-risk drug combinations: {high_risk}')

In [ ]:
# Generate drug safety report
from pathlib import Path
from pyhealth_enterprise.pipelines.report_generator import ReportGenerator
from pyhealth_enterprise.config import settings

report_gen = ReportGenerator()
report_gen.generate_drug_safety_report(
    interactions,
    settings.PROJECT_ROOT / 'data' / 'processed' / 'drug_safety_report.html'
)
print('Drug safety report generated')